# 11 Deep Learning (PyTorch): Predicting Infection with a Neural Network

A colleague asks: "Should we try deep learning?"
We'll build a simple binary classification network with PyTorch and see how far we can get on 280 rows of data.

Workflow: **data preprocessing → model architecture → training loop → early stopping → AUC evaluation → learning curve → comparison with sklearn**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: Data preprocessing (converting to tensors by hand) ---
import pathlib

import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -- CJK font setup (prevents Chinese labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Same features as Ch10 (no symptoms, to avoid data leakage)
num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

# One-hot encode categorical features
X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
X_np = X_df.values.astype(np.float32)
y_np = df["infected"].values.astype(np.float32)

# Standardize age
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

# 70/30 split
idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

X_train = torch.tensor(X_np[train_idx])
y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)
X_val = torch.tensor(X_np[val_idx])
y_val = torch.tensor(y_np[val_idx]).unsqueeze(1)

print(f"Feature dimensions: {X_train.shape[1]}")
print(f"Training set: {len(X_train)}, validation set: {len(X_val)}")
print(f"Feature names: {list(X_df.columns)}")

In [ ]:
# --- Step 2: Model architecture ---
# input_dim → 32 → 16 → 1
input_dim = X_train.shape[1]

model = nn.Sequential(
    nn.Linear(input_dim, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# Number of parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model architecture: {input_dim} → 32 → 16 → 1")
print(f"Total parameters: {n_params}")
print(f"Parameter / sample ratio: {n_params / len(X_train):.1f}")
print(f"\n→ More parameters than samples → very high overfitting risk!")

In [ ]:
# --- Step 3: Training loop + early stopping ---
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Record history
train_losses, val_losses = [], []
best_val_loss = float("inf")
patience, counter = 15, 0
best_state = None
best_epoch = 0

for epoch in range(300):
    # Train
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    # Validate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val).item()
    val_losses.append(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# Load the best model
model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}, best val_loss: {best_val_loss:.4f}")

In [ ]:
# --- Step 4: Learning curve ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train Loss", color="#2c7fb8")
ax.plot(val_losses, label="Val Loss", color="#e34a33")
ax.axvline(x=best_epoch, color="gray", linestyle="--", alpha=0.5,
           label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.set_title("Learning Curve")
ax.legend()
plt.tight_layout()
plt.show()

print("→ If train loss keeps falling but val loss rebounds → overfitting")
print("→ Early stopping halts training when val loss stops improving")

In [ ]:
# --- Step 5: AUC evaluation ---
model.eval()
with torch.no_grad():
    val_proba = torch.sigmoid(model(X_val)).numpy()
    train_proba = torch.sigmoid(model(X_train)).numpy()

auc_train = roc_auc_score(y_train.numpy(), train_proba)
auc_val = roc_auc_score(y_val.numpy(), val_proba)

print(f"=== PyTorch DL results ===")
print(f"Train AUC = {auc_train:.3f}")
print(f"Val   AUC = {auc_val:.3f}")
print(f"Gap       = {auc_train - auc_val:.3f}")

if auc_train - auc_val > 0.1:
    print("\n→ Train-Val gap > 0.1 → severe overfitting")
    print("→ 280 rows aren't enough to support this model's parameter count")
else:
    print("\n→ Gap is small; the model is relatively stable")

In [ ]:
# --- Step 6: Comparison with sklearn ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Use the same train/val split
X_full = df[num_cols + cat_cols + bin_cols]
y_full = df["infected"]

X_sk_train = X_full.iloc[train_idx]
X_sk_val = X_full.iloc[val_idx]
y_sk_train = y_full.iloc[train_idx]
y_sk_val = y_full.iloc[val_idx]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

compare = []

# Logistic Regression
clf_lr = Pipeline([("pre", preprocess), ("model", LogisticRegression(max_iter=500, random_state=42))])
clf_lr.fit(X_sk_train, y_sk_train)
auc_lr = roc_auc_score(y_sk_val, clf_lr.predict_proba(X_sk_val)[:, 1])
compare.append(("Logistic Regression", auc_lr))

# Random Forest
clf_rf = Pipeline([("pre", preprocess), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
clf_rf.fit(X_sk_train, y_sk_train)
auc_rf = roc_auc_score(y_sk_val, clf_rf.predict_proba(X_sk_val)[:, 1])
compare.append(("Random Forest", auc_rf))

# PyTorch
compare.append(("PyTorch DL", auc_val))

print("=== Model comparison (same train/val split) ===")
for name, auc in compare:
    print(f"  {name:25s}  Val AUC = {auc:.3f}")

print("\n→ On 280 rows, the three models usually perform very similarly")
print("→ DL shows no clear advantage and instead carries overfitting risk")
print("→ Educational value: learn PyTorch syntax so it pays off later on large datasets")

## Summary

| Step | Skill learned |
|------|------------|
| Preprocessing | `pd.get_dummies()` + `torch.tensor()` manual conversion |
| Model | `nn.Sequential(Linear → ReLU → Linear → ReLU → Linear)` |
| Training loop | `zero_grad → forward → loss → backward → step` |
| Early stopping | Monitor val_loss; stop once patience runs out |
| Learning curve | Visualize train/val loss to diagnose overfitting |
| Model comparison | Fair comparison of DL vs sklearn on the same split |

**Bottom line**:
- 280 rows → DL is overkill; sklearn is enough
- But PyTorch syntax is worth learning: you'll need it later for images, sequences, and large samples
- The point isn't "which model is strongest" but "using the right tool for the right problem"

In the next chapter (Ch12), we ask: did showering really "cause" the infections? → Causal inference.